In [ ]:
## ============================================
# MODEL CREATION
# ============================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b2
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from torch.cuda.amp import autocast, GradScaler

# -------------------
# CBAM Block (Attention)
# -------------------
class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super(CBAMBlock, self).__init__()
        # Channel Attention
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        self.sigmoid_channel = nn.Sigmoid()
        # Spatial Attention
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        # Channel Attention
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        channel_out = self.sigmoid_channel(avg_out + max_out).view(b, c, 1, 1)
        x = x * channel_out
        # Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        spatial_out = torch.cat([avg_out, max_out], dim=1)
        spatial_out = self.sigmoid_spatial(self.conv_spatial(spatial_out))
        x = x * spatial_out
        return x

# ---------------------------------------------------------
# Basic Conv Block
# ---------------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel=3, dropout=0.3):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel, padding=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.dropout(x)
        return x


# ---------------------------------------------------------
# Residual Block (2 conv blocks + skip)
# ---------------------------------------------------------
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = ConvBlock(channels, channels)
        self.conv2 = ConvBlock(channels, channels)

    def forward(self, x):
        identity = x
        x = self.conv1(x)
        x = self.conv2(x)
        return x + identity


# ---------------------------------------------------------
# BlockTwoSkip (2 residual blocks + skip)
# ---------------------------------------------------------
class BlockTwoSkip(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.res1 = ResidualBlock(channels)
        self.res2 = ResidualBlock(channels)

    def forward(self, x):
        identity = x
        x = self.res1(x)
        x = self.res2(x)
        return x + identity

class GraphConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj):
        # x: [B, N, C]
        # adj: [N, N]
        out = torch.matmul(adj, x)          
        out = self.linear(out)
        return F.relu(out)

def build_grid_adj(h, w, device):
    n = h * w
    adj = torch.zeros(n, n, device=device)

    for i in range(h):
        for j in range(w):
            idx = i * w + j
            for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < h and 0 <= nj < w:
                    nidx = ni * w + nj
                    adj[idx, nidx] = 1

    # self-loop + normalization
    adj += torch.eye(n, device=device)
    deg = adj.sum(dim=1, keepdim=True)
    adj = adj / deg
    return adj

# ---------------------------------------------------------
# Full Model with densenet121 Backbone
# ---------------------------------------------------------
class FERModel(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        # -------- DenseNet Backbone --------
        backbone = models.densenet121(
            weights=models.DenseNet121_Weights.IMAGENET1K_V1
        )
        self.backbone = backbone.features   # [B, 1024, 7, 7]
        in_channels = 1024


        # Feature blocks (كما هي)
        self.conv1a = ConvBlock(in_channels, 128)
        mid_channels = 128   

        # -------------------
        # Feature extractor
        # -------------------
        self.conv1 = ConvBlock(in_channels, mid_channels, dropout=0.2)
        self.block1 = BlockTwoSkip(mid_channels)

        self.conv2 = ConvBlock(mid_channels, mid_channels, dropout=0.2)
        self.block2 = BlockTwoSkip(mid_channels)

        self.cbam = CBAMBlock(mid_channels)

        self.conv3 = ConvBlock(mid_channels, mid_channels, dropout=0.2)
        self.block3 = BlockTwoSkip(mid_channels)

        self.gap = nn.AdaptiveAvgPool2d(1)
        # GNN
        
        self.gnn1 = GraphConv(mid_channels, mid_channels)
        self.gnn2 = GraphConv(mid_channels, mid_channels)
        
        self.gnn_fc = nn.Linear(mid_channels, mid_channels)
        # -------------------
        # Classifier
        # -------------------
        self.classifier = nn.Sequential(
            nn.Linear(mid_channels, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
    
        x = self.conv1(x)
        x = self.block1(x)
    
        x = self.conv2(x)
        x = self.block2(x)
    
        x = self.cbam(x)
        x = self.conv3(x)
        x = self.block3(x)
    
        # --------------------
        # CNN → Graph
        # --------------------
        B, C, H, W = x.shape
        nodes = x.view(B, C, H*W).permute(0, 2, 1)  # [B, N, C]
    
        adj = build_grid_adj(H, W, x.device)
    
        nodes = self.gnn1(nodes, adj)
        nodes = self.gnn2(nodes, adj)
    
        # Graph pooling
        gnn_feat = nodes.mean(dim=1)   # [B, C]
        gnn_feat = self.gnn_fc(gnn_feat)
    
        # --------------------
        # Final classifier
        # --------------------
        logits = self.classifier(gnn_feat)
        return logits


In [ ]:
BATCH_SIZE = 32             
LR = 3e-4                   
WEIGHT_DECAY = 1e-4
PATIENCE = 50               
FREEZE_EPOCHS = 10                       
LABEL_SMOOTHING = 0.1

In [ ]:
def freeze_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = False

def unfreeze_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = True


In [ ]:
class EarlyStopping:
    def __init__(self, patience=15):
        self.patience = patience
        self.counter = 0
        self.best_acc = 0

    def step(self, val_acc):
        if val_acc > self.best_acc:
            self.best_acc = val_acc
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience


In [ ]:
def set_seed(seed=42):
    import random, numpy as np, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
from torchvision import transforms

# ── CK+ Dataset Class ──────────────────────────────────────────────────────
class CKPlusDataset(Dataset):
    """
    هيكل مجلدات CK+:
    CK+/
    ├── angry/
    │   ├── S005_001_00000011.png
    │   └── ...
    ├── contempt/
    ├── disgust/
    ├── fear/
    ├── happy/
    ├── sadness/
    └── surprise/
    """

    EMOTIONS = ['anger', 'contempt', 'disgust', 'fear', 'happy', 'sadness', 'surprise']

    def __init__(self, root_dir, transform=None):
        self.root_dir  = root_dir
        self.transform = transform
        self.samples   = []   # list of (img_path, label)
        self.subjects  = []   # subject ID per sample (for LOSO option)

        for label, emotion in enumerate(self.EMOTIONS):
            emotion_dir = os.path.join(root_dir, emotion)
            if not os.path.isdir(emotion_dir):
                continue
            for fname in sorted(os.listdir(emotion_dir)):
                if fname.lower().endswith(('.png','.jpg','.jpeg')):
                    fpath = os.path.join(emotion_dir, fname)
                    self.samples.append((fpath, label))
                    # CK+ filenames: S005_001_... → subject = S005
                    subject = fname.split('_')[0]
                    self.subjects.append(subject)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
# ── Transforms ──────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
import torch

# -------------------
# Load dataset
# -------------------

transform  = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])


full_dataset = datasets.ImageFolder('D/ckplus/CK+48', transform=transform)
emotion_labels = full_dataset.classes
print("Emotion labels:", emotion_labels)

# -------------------
# Split dataset (80/10/10)
# -------------------
train_size = int(0.80 * len(full_dataset))
val_size   = int(0.20 * len(full_dataset))
test_size  = len(full_dataset) - train_size - val_size  # remaining samples

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)  # for reproducibility
)

# -------------------
# Create DataLoaders
# -------------------
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader  = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import numpy as np

def train_one_fold(model, train_loader, test_loader,
                   fold_num, device,
                   num_epochs=50, patience=15):
    """
    يدرّب ويقيّم fold واحد.
    Returns: best_acc, best_f1, all_preds, all_labels
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_acc     = 0.0
    best_f1      = 0.0
    best_preds   = []
    best_labels  = []
    no_improve   = 0

    print(f"\n{'='*50}")
    print(f"  Fold {fold_num} — Training")
    print(f"{'='*50}")

    for epoch in range(1, num_epochs + 1):

        # ── Train ────────────────────────────────────────
        model.train()
        train_loss, correct, total = 0.0, 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item() * imgs.size(0)
            preds       = outputs.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)

        scheduler.step()
        train_acc  = correct / total
        train_loss = train_loss / total

        # ── Evaluate ─────────────────────────────────────
        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs   = imgs.to(device)
                outputs = model(imgs)
                preds   = outputs.argmax(dim=1).cpu().tolist()
                all_preds  .extend(preds)
                all_labels .extend(labels.tolist())

        acc = accuracy_score(all_labels, all_preds) * 100
        f1  = f1_score(all_labels, all_preds,
                       average='weighted') * 100

        print(f"Epoch {epoch:03d}/{num_epochs} | "
              f"Loss={train_loss:.4f} | "
              f"TrainAcc={train_acc*100:.2f}% | "
              f"TestAcc={acc:.2f}% | F1={f1:.2f}%")

        # ── Early Stopping ────────────────────────────────
        if acc > best_acc:
            best_acc    = acc
            best_f1     = f1
            best_preds  = all_preds.copy()
            best_labels = all_labels.copy()
            no_improve  = 0
            # احفظ أحسن weights
            torch.save(model.state_dict(),
                       f'best_fold{fold_num}.pt')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  ⏹ Early stopping at epoch {epoch}")
                break

    print(f"\n  ✅ Fold {fold_num} Best Acc = {best_acc:.2f}% | "
          f"F1 = {best_f1:.2f}%")

    return best_acc, best_f1, best_preds, best_labels

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import os
import re
import numpy as np


def get_subject(path):
    """
    يستخرج subject ID من مسار الصورة.
    في CK+ الأسماء بتبقى زي: S005_001_00000011.png  ->  S005
    عدّل الـ regex لو تنظيم ملفاتك مختلف.
    """
    m = re.search(r'(S\d+)', os.path.basename(path))
    if m is None:
        # محاولة تانية على المسار كامل (لو الـ subject اسم فولدر)
        m = re.search(r'(S\d+)', path)
    if m is None:
        raise ValueError(f"Can't extract subject from: {path}")
    return m.group(1)


def run_5fold_ckplus(root_dir, device, num_classes=7):
    """
    يشغّل 5-Fold Cross-Validation (subject-independent) على CK+
    باستخدام StratifiedGroupKFold.
    """

    # ── Load full dataset ────────────────────────────────
    full_dataset = CKPlusDataset(root_dir, transform=train_transform)

    all_paths  = [full_dataset.samples[i][0] for i in range(len(full_dataset))]
    all_labels = [full_dataset.samples[i][1] for i in range(len(full_dataset))]
    groups     = [get_subject(p) for p in all_paths]

    print(f"📦 Total samples : {len(full_dataset)}")
    print(f"👤 Total subjects: {len(set(groups))}")
    print(f"📊 Classes: {CKPlusDataset.EMOTIONS}")

    # ── Stratified Group 5-Fold ──────────────────────────
    # Group      = نفس الشخص عمره ما يتقسم بين train و test
    # Stratified = بيحاول يحافظ على توزيع الكلاسات في كل fold
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

    fold_results = []   # (acc, f1) لكل fold
    all_preds_combined  = []
    all_labels_combined = []

    for fold, (train_idx, test_idx) in enumerate(
            sgkf.split(np.arange(len(full_dataset)), all_labels, groups=groups),
            start=1):

        # ── تأكيد: مفيش أي شخص مشترك بين train و test ──
        train_subjects = {groups[i] for i in train_idx}
        test_subjects  = {groups[i] for i in test_idx}
        assert train_subjects.isdisjoint(test_subjects), \
            f"Subject leakage in fold {fold}!"

        print(f"\n{'━'*60}")
        print(f"  FOLD {fold}/5 | "
              f"Train={len(train_idx)} ({len(train_subjects)} subj) | "
              f"Test={len(test_idx)} ({len(test_subjects)} subj)")
        print(f"{'━'*60}")

        # ── Datasets ─────────────────────────────────────
        # train dataset: بيستخدم train_transform
        # test dataset:  بيستخدم test_transform (بدون augmentation)
        train_dataset = CKPlusDataset(root_dir, transform=train_transform)
        test_dataset  = CKPlusDataset(root_dir, transform=test_transform)

        train_loader = DataLoader(
            train_dataset,
            batch_size=32,
            sampler=SubsetRandomSampler(train_idx),
            num_workers=2,
            pin_memory=True,
        )
        test_loader = DataLoader(
            test_dataset,
            batch_size=32,
            sampler=SubsetRandomSampler(test_idx),
            num_workers=2,
            pin_memory=True,
        )

        # ── Fresh model لكل fold ──────────────────────────
        model = FERModel(num_classes=num_classes).to(device)

        # ── Train ─────────────────────────────────────────
        acc, f1, preds, labels = train_one_fold(
            model, train_loader, test_loader,
            fold_num=fold,
            device=device,
            num_epochs=50,
            patience=50,
        )

        fold_results.append((acc, f1))
        all_preds_combined .extend(preds)
        all_labels_combined.extend(labels)

    # ────────────────────────────────────────────────────
    # النتائج النهائية
    # ────────────────────────────────────────────────────
    accs = [r[0] for r in fold_results]
    f1s  = [r[1] for r in fold_results]

    print(f"\n{'═'*60}")
    print(f"  5-FOLD SUBJECT-INDEPENDENT CV RESULTS — CK+")
    print(f"{'═'*60}")
    for i, (acc, f1) in enumerate(fold_results, 1):
        print(f"  Fold {i}: Acc={acc:.2f}%  F1={f1:.2f}%")
    print(f"{'─'*60}")
    print(f"  Mean Acc : {np.mean(accs):.2f}% ± {np.std(accs):.2f}%")
    print(f"  Mean F1  : {np.mean(f1s):.2f}% ± {np.std(f1s):.2f}%")
    print(f"  Best Fold: {np.argmax(accs)+1} ({max(accs):.2f}%)")
    print(f"{'═'*60}")

    # Confusion matrix على كل الـ predictions
    cm = confusion_matrix(all_labels_combined, all_preds_combined)
    print(f"\n  Confusion Matrix (all folds combined):")
    print(cm)

    return {
        'fold_results'    : fold_results,
        'mean_acc'        : np.mean(accs),
        'std_acc'         : np.std(accs),
        'mean_f1'         : np.mean(f1s),
        'std_f1'          : np.std(f1s),
        'confusion_matrix': cm,
    }

In [ ]:
import os

root = "D/ckplus/CK+48"
exts = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

files = [os.path.join(d, f)
         for d, _, fs in os.walk(root)
         for f in fs if f.lower().endswith(exts)]
print("Total images:", len(files))

In [ ]:
if __name__ == "__main__":
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    root_dir = "D/ckplus/CK+48"  # ← غيّر المسار

    results = run_5fold_ckplus(root_dir, device, num_classes=7)